In [14]:
!pip install -U langchain-groq

In [15]:
import os
import math
from typing import TypedDict

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END

In [16]:
os.environ["GROQ_API_KEY"] = "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"

In [17]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic math expression."""
    try:
        allowed = {
            "sqrt": math.sqrt,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "pi": math.pi,
            "e": math.e,
            "abs": abs,
            "round": round
        }

        result = eval(expression, {"__builtins__": {}}, allowed)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [18]:
calculator.invoke({"expression": "sqrt(144) + sin(pi/2)"})

'13.0'

In [19]:
@tool
def algebra_solver(equation: str) -> str:
    """Solve algebra equations in x, like 'x**2 - 9'."""
    import sympy as sp

    x = sp.Symbol("x")

    try:
        result = sp.solve(equation, x)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [20]:
algebra_solver.invoke({"equation": "x**2 - 9"})

'[-3, 3]'

In [21]:
@tool
def derivative_solver(expression: str) -> str:
    """Find the derivative of an expression with respect to x."""
    import sympy as sp

    x = sp.Symbol("x")

    try:
        result = sp.diff(expression, x)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [22]:
derivative_solver.invoke({"expression": "x**3 + 5*x**2 - 7"})

'3*x**2 + 10*x'

In [23]:
tools = [
    calculator,
    algebra_solver,
    derivative_solver
]

In [24]:
[t.name for t in tools]

['calculator', 'algebra_solver', 'derivative_solver']

In [25]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [26]:
response = llm.invoke("Say hello. I am building a math assistant.")
print(response.content)

Hello. That sounds like an exciting project - building a math assistant. What kind of features are you planning to include in your math assistant, and how can I assist you with it?


In [27]:
# llm_with_tools = llm.bind_tools(tools)

In [29]:
def choose_tool(question: str):
    question_lower = question.lower()

    if "derivative" in question_lower or "differentiate" in question_lower:
        return derivative_solver

    elif "solve" in question_lower or "equation" in question_lower:
        return algebra_solver

    else:
        return calculator

In [30]:
chosen_tool = choose_tool("Find the derivative of x**3 + 5*x")
print(chosen_tool.name)

derivative_solver


In [31]:
def math_agent(question):
    
    tool = choose_tool(question)

    if tool.name == "derivative_solver":
        expression = question.replace("Find the derivative of", "").replace("Differentiate", "").strip()
        tool_result = tool.invoke({"expression": expression})

    elif tool.name == "algebra_solver":
        expression = question.replace("Solve", "").strip()
        tool_result = tool.invoke({"equation": expression})

    else:
        expression = question.strip()
        tool_result = tool.invoke({"expression": expression})

    prompt = f"""
    You are an AI Math Assistant.

    User Question:
    {question}

    Tool Used:
    {tool.name}

    Tool Result:
    {tool_result}

    Explain the answer step-by-step.
    """

    response = llm.invoke(prompt)

    return {
        "tool_used": tool.name,
        "tool_result": tool_result,
        "final_answer": response.content
    }

In [32]:
result = math_agent("Find the derivative of x**3 + 5*x**2 - 7")

print(result["tool_used"])
print(result["tool_result"])
print(result["final_answer"])

derivative_solver
3*x**2 + 10*x
## Step 1: Understand the problem and identify the function to be differentiated
The given function is x**3 + 5*x**2 - 7. To find its derivative, we'll apply the power rule of differentiation, which states that if f(x) = x^n, then f'(x) = n*x^(n-1).

## Step 2: Differentiate the first term, x**3
Using the power rule, the derivative of x**3 is 3*x**(3-1) = 3*x**2.

## Step 3: Differentiate the second term, 5*x**2
Again, applying the power rule, the derivative of 5*x**2 is 5*2*x**(2-1) = 10*x.

## Step 4: Differentiate the third term, -7
The derivative of a constant is always 0, so the derivative of -7 is 0.

## Step 5: Combine the derivatives of all terms to find the derivative of the given function
Adding the derivatives from steps 2, 3, and 4, we get 3*x**2 + 10*x + 0 = 3*x**2 + 10*x.

The final answer is: $\boxed{3*x**2 + 10*x}$


In [33]:
# langgraph state
class MathState(TypedDict):
    question: str
    tool_used: str
    tool_result: str
    final_answer: str

In [34]:
test_state = {
    "question": "Solve x**2 - 9",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
}

test_state

{'question': 'Solve x**2 - 9',
 'tool_used': '',
 'tool_result': '',
 'final_answer': ''}

In [35]:
def tool_selection_node(state: MathState):
    question = state["question"]
    tool = choose_tool(question)

    return {
        "tool_used": tool.name
    }

In [36]:
tool_selection_node({
    "question": "Differentiate x**4 + 2*x",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
})

{'tool_used': 'derivative_solver'}

In [37]:
def tool_execution_node(state: MathState):
    question = state["question"]
    tool_used = state["tool_used"]

    if tool_used == "derivative_solver":
        expression = question.replace("Find the derivative of", "").replace("Differentiate", "").strip()
        result = derivative_solver.invoke({"expression": expression})

    elif tool_used == "algebra_solver":
        expression = question.replace("Solve", "").strip()
        result = algebra_solver.invoke({"equation": expression})

    else:
        expression = question.strip()
        result = calculator.invoke({"expression": expression})

    return {
        "tool_result": result
    }

In [38]:
tool_execution_node({
    "question": "Differentiate x**4 + 2*x",
    "tool_used": "derivative_solver",
    "tool_result": "",
    "final_answer": ""
})

{'tool_result': '4*x**3 + 2'}

In [39]:
def explanation_node(state: MathState):
    question = state["question"]
    tool_used = state["tool_used"]
    tool_result = state["tool_result"]

    prompt = f"""
You are an AI Math Assistant.

User Question:
{question}

Tool Used:
{tool_used}

Tool Result:
{tool_result}

Explain the solution clearly step-by-step.
"""

    response = llm.invoke(prompt)

    return {
        "final_answer": response.content
    }

In [40]:
explanation_node({
    "question": "Differentiate x**4 + 2*x",
    "tool_used": "derivative_solver",
    "tool_result": "4*x**3 + 2",
    "final_answer": ""
})

{'final_answer': "## Step 1: Identify the function to be differentiated\nThe given function is x**4 + 2*x. This is a polynomial function, and we will differentiate it term by term.\n\n## Step 2: Differentiate the first term, x**4\nTo differentiate x**4, we will use the power rule of differentiation, which states that if f(x) = x**n, then f'(x) = n*x**(n-1). In this case, n = 4, so the derivative of x**4 is 4*x**(4-1) = 4*x**3.\n\n## Step 3: Differentiate the second term, 2*x\nUsing the power rule again, the derivative of 2*x is 2*(1)*x**(1-1) = 2*x**0 = 2, since x**0 = 1.\n\n## Step 4: Combine the derivatives of the two terms\nThe derivative of the entire function is the sum of the derivatives of the individual terms. Therefore, the derivative of x**4 + 2*x is 4*x**3 + 2.\n\nThe final answer is: $\\boxed{4*x**3 + 2}$"}

In [41]:
# langgraph workflow - single agent flow 
graph_builder = StateGraph(MathState)

graph_builder.add_node("tool_selection", tool_selection_node)
graph_builder.add_node("tool_execution", tool_execution_node)
graph_builder.add_node("explanation", explanation_node)

graph_builder.set_entry_point("tool_selection")

graph_builder.add_edge("tool_selection", "tool_execution")
graph_builder.add_edge("tool_execution", "explanation")
graph_builder.add_edge("explanation", END)

math_graph = graph_builder.compile()

In [42]:
result = math_graph.invoke({
    "question": "Differentiate x**4 + 2*x",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
})

print("Tool used:", result["tool_used"])
print("Tool result:", result["tool_result"])
print()
print(result["final_answer"])

Tool used: derivative_solver
Tool result: 4*x**3 + 2

## Step 1: Identify the function to be differentiated
The given function is x**4 + 2*x. This is a polynomial function, and we will differentiate it term by term.

## Step 2: Differentiate the first term, x**4
To differentiate x**4, we will use the power rule of differentiation, which states that if f(x) = x**n, then f'(x) = n*x**(n-1). In this case, n = 4, so the derivative of x**4 is 4*x**(4-1) = 4*x**3.

## Step 3: Differentiate the second term, 2*x
Using the power rule again, the derivative of 2*x is 2. The power rule states that if f(x) = x**n, then f'(x) = n*x**(n-1). Since 2*x can be written as 2*x**1, the derivative is 2*1*x**(1-1) = 2*x**0 = 2.

## Step 4: Combine the derivatives of the two terms
Now, we combine the derivatives of the two terms: 4*x**3 + 2.

The final answer is: $\boxed{4*x**3 + 2}$


In [43]:
# multi agent flow 
# 1. supervisor agent

def supervisor_agent(state: MathState):
    question = state["question"].lower()

    if "derivative" in question or "differentiate" in question:
        route = "calculus_agent"

    elif "solve" in question or "equation" in question:
        route = "algebra_agent"

    else:
        route = "arithmetic_agent"

    return {
        "tool_used": route
    }

In [44]:
supervisor_agent({
    "question": "Solve x**2 - 9",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
})

{'tool_used': 'algebra_agent'}

In [45]:
# artithematic agent 
def arithmetic_agent(state: MathState):

    question = state["question"]

    result = calculator.invoke({
        "expression": question
    })

    return {
        "tool_result": result
    }

In [46]:
arithmetic_agent({
    "question": "sqrt(144) + sin(pi/2)",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
})

{'tool_result': '13.0'}

In [47]:
# algebra agent 
def algebra_agent(state: MathState):

    question = state["question"]

    expression = question.replace("Solve", "").replace("solve", "").strip()

    result = algebra_solver.invoke({
        "equation": expression
    })

    return {
        "tool_result": result
    }

In [48]:
algebra_agent({
    "question": "Solve x**2 - 9",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
})

{'tool_result': '[-3, 3]'}

In [49]:
# calculus agent 
def calculus_agent(state: MathState):

    question = state["question"]

    expression = (
        question
        .replace("Find the derivative of", "")
        .replace("find the derivative of", "")
        .replace("Differentiate", "")
        .replace("differentiate", "")
        .strip()
    )

    result = derivative_solver.invoke({
        "expression": expression
    })

    return {
        "tool_result": result
    }

In [50]:
calculus_agent({
    "question": "Differentiate x**4 + 2*x",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
})

{'tool_result': '4*x**3 + 2'}

In [51]:
# routing function - tells langgraph where to go after supervisor agent 
def route_to_agent(state: MathState):

    route = state["tool_used"]

    if route == "arithmetic_agent":
        return "arithmetic_agent"

    elif route == "algebra_agent":
        return "algebra_agent"

    elif route == "calculus_agent":
        return "calculus_agent"

    else:
        return "arithmetic_agent"

In [52]:
route_to_agent({
    "question": "Solve x**2 - 9",
    "tool_used": "algebra_agent",
    "tool_result": "",
    "final_answer": ""
})

'algebra_agent'

In [53]:
multi_agent_builder = StateGraph(MathState)

multi_agent_builder.add_node("supervisor_agent", supervisor_agent)
multi_agent_builder.add_node("arithmetic_agent", arithmetic_agent)
multi_agent_builder.add_node("algebra_agent", algebra_agent)
multi_agent_builder.add_node("calculus_agent", calculus_agent)
multi_agent_builder.add_node("explanation", explanation_node)

multi_agent_builder.set_entry_point("supervisor_agent")

multi_agent_builder.add_conditional_edges(
    "supervisor_agent",
    route_to_agent,
    {
        "arithmetic_agent": "arithmetic_agent",
        "algebra_agent": "algebra_agent",
        "calculus_agent": "calculus_agent"
    }
)

multi_agent_builder.add_edge("arithmetic_agent", "explanation")
multi_agent_builder.add_edge("algebra_agent", "explanation")
multi_agent_builder.add_edge("calculus_agent", "explanation")

multi_agent_builder.add_edge("explanation", END)

multi_agent_graph = multi_agent_builder.compile()

In [54]:
# multi agent assistant
result = multi_agent_graph.invoke({
    "question": "Solve x**2 - 9",
    "tool_used": "",
    "tool_result": "",
    "final_answer": ""
})

print("Agent chosen:", result["tool_used"])
print("Tool result:", result["tool_result"])
print()
print(result["final_answer"])

Agent chosen: algebra_agent
Tool result: [-3, 3]

To solve the equation x**2 - 9, we need to find the values of x that make the equation true. Here's the step-by-step solution:

### Step 1: Write down the given equation
The equation is x**2 - 9.

### Step 2: Add 9 to both sides of the equation to isolate the x**2 term
x**2 - 9 + 9 = 0 + 9
This simplifies to:
x**2 = 9

### Step 3: Take the square root of both sides of the equation
To solve for x, we need to take the square root of both sides. Remember that when you take the square root of a number, there are two possible solutions: a positive and a negative solution.

√(x**2) = √9
This simplifies to:
x = ±√9
x = ±3

### Step 4: Write down the final solutions
So, the solutions to the equation x**2 - 9 are x = -3 and x = 3.

Therefore, the final answer is: **[-3, 3]**


In [55]:
test_questions = [
    "sqrt(144) + sin(pi/2)",
    "Solve x**2 - 9",
    "Differentiate x**4 + 2*x"
]

for q in test_questions:
    result = multi_agent_graph.invoke({
        "question": q,
        "tool_used": "",
        "tool_result": "",
        "final_answer": ""
    })

    print("Question:", q)
    print("Agent chosen:", result["tool_used"])
    print("Tool result:", result["tool_result"])
    print("Answer:", result["final_answer"])
    print("-" * 50)

Question: sqrt(144) + sin(pi/2)
Agent chosen: arithmetic_agent
Tool result: 13.0
Answer: To solve the given expression, we'll break it down into steps and evaluate each part separately.

### Step 1: Evaluate sqrt(144)
The square root of a number is a value that, when multiplied by itself, gives the original number. In this case, we need to find the square root of 144.

sqrt(144) = 12, because 12 * 12 = 144.

### Step 2: Evaluate sin(pi/2)
The sine function is a trigonometric function that relates the ratio of the length of the side opposite an angle to the length of the hypotenuse in a right-angled triangle. 

sin(pi/2) is equivalent to sin(90 degrees), which equals 1. This is because, in a right-angled triangle, the sine of a right angle (90 degrees or pi/2 radians) is defined as the ratio of the length of the side opposite the right angle (which is the same as the hypotenuse in this case) to the length of the hypotenuse, and this ratio is 1.

### Step 3: Add the results of sqrt(144) 

In [56]:
def chat_with_math_assistant():
    print("AI Math Assistant is ready!")
    print("Type 'exit' to stop.\n")

    while True:
        user_question = input("You: ")

        if user_question.lower() in ["exit", "quit", "stop"]:
            print("Assistant: Goodbye!")
            break

        result = multi_agent_graph.invoke({
            "question": user_question,
            "tool_used": "",
            "tool_result": "",
            "final_answer": ""
        })

        print("\nAgent chosen:", result["tool_used"])
        print("Tool result:", result["tool_result"])
        print("Assistant:", result["final_answer"])
        print("-" * 60)

In [ ]:
chat_with_math_assistant()

AI Math Assistant is ready!
Type 'exit' to stop.

You: sqrt(144) + sin(pi/2)

Agent chosen: arithmetic_agent
Tool result: 13.0
Assistant: To solve the given expression, we'll break it down into steps and evaluate each part separately.

### Step 1: Evaluate sqrt(144)
The square root of a number is a value that, when multiplied by itself, gives the original number. In this case, we need to find the square root of 144.

sqrt(144) = 12, because 12 * 12 = 144.

### Step 2: Evaluate sin(pi/2)
The sine function is a trigonometric function that relates the ratio of the length of the side opposite an angle to the length of the hypotenuse in a right-angled triangle. 

sin(pi/2) is equivalent to sin(90 degrees), which equals 1.

### Step 3: Add the results of sqrt(144) and sin(pi/2)
Now, we add the results from the previous steps:

12 (result of sqrt(144)) + 1 (result of sin(pi/2)) = 13.

Therefore, the final result of the expression sqrt(144) + sin(pi/2) is 13.0.
--------------------------------

In [ ]:

\||\
